In [53]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

In [54]:
# loading the data

df = pd.read_csv('../data/processed/clean_bmw_price.csv')

df.head()

,mileage,engine_power,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,fuel_petrol,paint_color_rare,paint_color_standard,car_type_coupe,car_type_estate,car_type_hatchback,car_type_sedan,car_type_subcompact,car_type_suv,car_type_van
0,140411,100,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,13929,317,1,1,0,0,0,1,1,1,...,1,0,0,0,0,0,0,0,0,0
2,183297,120,0,0,0,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,128035,135,1,1,0,0,1,1,1,1,...,0,1,0,0,0,0,0,0,0,0
4,97097,160,1,1,0,0,0,1,1,1,...,0,0,1,0,0,0,0,0,0,0


In [55]:
# splitting the model

X = df.drop(columns='cube_root_price')
y = df['cube_root_price'].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'X Train shape: {X_train.shape}')
print(f'X Test shape: {X_test.shape}')
print(f'y Train shape: {y_train.shape}')
print(f'y Test shape: {y_test.shape}')

X Train shape: (3872, 23)
X Test shape: (969, 23)
y Train shape: (3872,)
y Test shape: (969,)


In [56]:
# Training the base model Linear Regression

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_prediction = lr.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, lr_prediction): 0.2f}')

The R2 Score for LR:  0.75


In [57]:
# Convert predictions and test data back to original scale
y_test_actual = np.power(y_test, 3)
y_pred_actual = np.power(lr_prediction, 3)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2430.49
Average Percentage Error: 62.43%


In [58]:
# Training the Random Forest Model

rf = RandomForestRegressor()

rf.fit(X_train, y_train)

rf_prediction = rf.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, rf_prediction): 0.2f}')

The R2 Score for LR:  0.76


In [59]:
# Convert predictions and test data back to original scale
y_test_actual = np.power(y_test, 3)
y_pred_actual = np.power(rf_prediction, 3)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2228.90
Average Percentage Error: 48.59%


In [60]:
# Random CV for better parameters for Random forest

params = {
    'n_estimators': [100,200,200,500],
    'max_depth': [10,20,30,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'bootstrap': [True, False]
}

random = RandomizedSearchCV(
    estimator=RandomForestRegressor(),
    param_distributions=params,
    n_iter=20,
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_absolute_error'
)

random.fit(X_train, y_train)

print(random.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20, 'bootstrap': True}


In [61]:
# Training Random Forest with tuned hyper parameters

rf__final = RandomForestRegressor(
    n_estimators=200,
    min_samples_split= 5,
    min_samples_leaf= 1,
    max_depth=20,
    bootstrap=True
)

rf__final.fit(X_train, y_train)

rf_final_prediction = rf__final.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, rf_final_prediction): 0.2f}')

The R2 Score for LR:  0.77


In [62]:
# Convert predictions and test data back to original scale
y_test_actual = np.power(y_test, 3)
y_pred_actual = np.power(rf_final_prediction, 3)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2215.68
Average Percentage Error: 50.23%


# Conclusion
Both the models failed at giving a low MAPE, need to change the target column from cuberoot to log and then train again

In [63]:
df['price'] = np.power(df['cube_root_price'],3)

df.head()

,mileage,engine_power,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,paint_color_rare,paint_color_standard,car_type_coupe,car_type_estate,car_type_hatchback,car_type_sedan,car_type_subcompact,car_type_suv,car_type_van,price
0,140411,100,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,11300.0
1,13929,317,1,1,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,69700.0
2,183297,120,0,0,0,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,10200.0
3,128035,135,1,1,0,0,1,1,1,1,...,1,0,0,0,0,0,0,0,0,25100.0
4,97097,160,1,1,0,0,0,1,1,1,...,0,1,0,0,0,0,0,0,0,33400.0


In [64]:
# dropping the cube root column

df = df.drop(columns='cube_root_price')

In [65]:
# Cap the extreme ends of the price spectrum

lower_limit = df['price'].quantile(0.02)
upper_limit = df['price'].quantile(0.98)
df['price']= df['price'].clip(lower=lower_limit, upper=upper_limit)

df['price'].skew()

np.float64(1.220465974894623)

In [66]:
# converting the price to log

df['log_price'] = np.log1p(df['price'])

df.head()

,mileage,engine_power,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,paint_color_standard,car_type_coupe,car_type_estate,car_type_hatchback,car_type_sedan,car_type_subcompact,car_type_suv,car_type_van,price,log_price
0,140411,100,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,11300.0,9.332646
1,13929,317,1,1,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,41220.0,10.626703
2,183297,120,0,0,0,0,1,0,1,0,...,1,0,0,0,0,0,0,0,10200.0,9.230241
3,128035,135,1,1,0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,25100.0,10.130663
4,97097,160,1,1,0,0,0,1,1,1,...,1,0,0,0,0,0,0,0,33400.0,10.416341


In [67]:
# splitting the data 

X = df.drop(columns=['price', 'log_price'])
y = df['log_price'].values.ravel()



In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'X Train shape: {X_train.shape}')
print(f'X Test shape: {X_test.shape}')
print(f'y Train shape: {y_train.shape}')
print(f'y Test shape: {y_test.shape}')

X Train shape: (3872, 23)
X Test shape: (969, 23)
y Train shape: (3872,)
y Test shape: (969,)


In [69]:
# Training the base model Linear Regression

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_prediction = lr.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, lr_prediction): 0.2f}')

The R2 Score for LR:  0.75


In [70]:
# Convert predictions and test data back to original scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(lr_prediction)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2344.93
Average Percentage Error: 20.89%


In [73]:
# Training the Random Forest Model

rf = RandomForestRegressor()

rf.fit(X_train, y_train)

rf_prediction = rf.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, rf_prediction): 0.2f}')

The R2 Score for LR:  0.76


In [74]:
# Convert predictions and test data back to original scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(rf_prediction)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2086.82
Average Percentage Error: 18.35%


In [75]:
# Random CV for better parameters for Random forest

params = {
    'n_estimators': [100,200,200,500],
    'max_depth': [10,20,30,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'bootstrap': [True, False]
}

random = RandomizedSearchCV(
    estimator=RandomForestRegressor(),
    param_distributions=params,
    n_iter=20,
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_absolute_error'
)

random.fit(X_train, y_train)

print(random.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20, 'bootstrap': True}


In [76]:
# Training Random Forest with tuned hyper parameters

rf__final = RandomForestRegressor(
    n_estimators=200,
    min_samples_split= 5,
    min_samples_leaf= 1,
    max_depth=20,
    bootstrap=True
)

rf__final.fit(X_train, y_train)

rf_final_prediction = rf__final.predict(X_test)

print(f'The R2 Score for LR: {r2_score(y_test, rf_final_prediction): 0.2f}')

The R2 Score for LR:  0.76


In [77]:
# Convert predictions and test data back to original scale
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(rf_final_prediction)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = mean_absolute_percentage_error(y_test_actual, y_pred_actual)

print(f"Average Error in Actual Price: {mae: 0.2f}")
print(f'Average Percentage Error: {mape*100:.2f}%')

Average Error in Actual Price:  2083.74
Average Percentage Error: 18.35%
